Start by importing the required packages.

In [2]:
import re
import pandas as pd
import numpy as np
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

Load in a CSV file and look and the column names and datatypes.

In [4]:
atp2023 = pd.read_csv('atp_matches_2023.csv')
atp2023.dtypes

tourney_id             object
tourney_name           object
surface                object
draw_size               int64
tourney_level          object
tourney_date            int64
match_num               int64
winner_id               int64
winner_seed           float64
winner_entry           object
winner_name            object
winner_hand            object
winner_ht             float64
winner_ioc             object
winner_age            float64
loser_id                int64
loser_seed            float64
loser_entry            object
loser_name             object
loser_hand             object
loser_ht              float64
loser_ioc              object
loser_age             float64
score                  object
best_of                 int64
round                  object
minutes               float64
w_ace                 float64
w_df                  float64
w_svpt                float64
w_1stIn               float64
w_1stWon              float64
w_2ndWon              float64
w_SvGms   

Drop unnecessary columns, change the date column to datetime, and list unique values in the tourney_name column.

In [6]:
atp2023 = atp2023[["tourney_id", "tourney_name", "surface", "draw_size", "tourney_level", "tourney_date"]]
atp2023['tourney_date'] = pd.to_datetime(atp2023['tourney_date'].astype(str), format='%Y%m%d')
atp2023.dtypes

tourney_id               object
tourney_name             object
surface                  object
draw_size                 int64
tourney_level            object
tourney_date     datetime64[ns]
dtype: object

In [7]:
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals', 'Davis Cup

Remove the values that contain "Davis Cup". These won't be used in the analysis.

In [9]:
atp2023 = atp2023[~atp2023['tourney_name'].str.contains('Davis Cup', case=False, na=False)]
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells Masters', 'Miami Masters', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo Masters', 'Barcelona', 'Munich',
       'Banja Luka', 'Madrid Masters', 'Rome Masters', 'Geneva', 'Lyon',
       'Roland Garros', 's Hertogenbosch', 'Stuttgart', 'Halle',
       "Queen's Club", 'Mallorca', 'Eastbourne', 'Wimbledon', 'Gstaad',
       'Bastad', 'Newport', 'Atlanta', 'Hamburg', 'Umag', 'Kitzbuhel',
       'Los Cabos', 'Washington', 'Canada Masters', 'Cincinnati Masters',
       'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai', 'Astana',
       'Beijing', 'Shanghai Masters', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris Masters', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=obj

Remove "Masters" from any of the values. This will help find the locations.

In [11]:
atp2023["tourney_name"] = atp2023["tourney_name"].str.replace("masters", "", case=False, regex=True).str.strip()
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Australian Open', 'Cordoba', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai', 'Santiago',
       'Indian Wells', 'Miami', 'Estoril', 'Houston', 'Marrakech',
       'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka', 'Madrid',
       'Rome', 'Geneva', 'Lyon', 'Roland Garros', 's Hertogenbosch',
       'Stuttgart', 'Halle', "Queen's Club", 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos', 'Washington', 'Canada',
       'Cincinnati', 'Winston-Salem', 'Us Open', 'Chengdu', 'Zhuhai',
       'Astana', 'Beijing', 'Shanghai', 'Stockholm', 'Tokyo', 'Antwerp',
       'Basel', 'Vienna', 'Paris', 'Metz', 'Sofia', 'Tour Finals',
       'NextGen Finals'], dtype=object)

Change the tournament names without clear locations to be more specific.

In [13]:
atp2023["tourney_name"] = atp2023["tourney_name"].replace({
    "Australian Open": "Melbourne",
    "Roland Garros": "Paris",
    "Queen's Club": "London",
    "Canada": "Toronto",
    "Us Open": "New York",
    "Tour Finals": "Turin",
    "NextGen Finals": "Jeddah",
    "Cordoba": "Cordoba, Argentina",
    "Los Cabos": "Los Cabos, Baja California Sur, Mexico",
    "Santiago": "Santiago, Chile",
})
atp2023["tourney_name"].unique()

array(['United Cup', 'Adelaide 1', 'Pune', 'Auckland', 'Adelaide 2',
       'Melbourne', 'Cordoba, Argentina', 'Dallas', 'Montpellier',
       'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Baja California Sur, Mexico',
       'Washington', 'Toronto', 'Cincinnati', 'Winston-Salem', 'New York',
       'Chengdu', 'Zhuhai', 'Astana', 'Beijing', 'Shanghai', 'Stockholm',
       'Tokyo', 'Antwerp', 'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin',
       'Jeddah'], dtype=object)

Change "United Cup" to the locations where the event is held.

In [15]:
atp2023["tourney_name"] = atp2023["tourney_name"].apply(
    lambda x: ["Brisbane", "Sydney", "Perth"] if x == "United Cup" else [x]
)

atp2023 = atp2023.explode("tourney_name")
atp2023["tourney_name"].unique()

array(['Brisbane', 'Sydney', 'Perth', 'Adelaide 1', 'Pune', 'Auckland',
       'Adelaide 2', 'Melbourne', 'Cordoba, Argentina', 'Dallas',
       'Montpellier', 'Delray Beach', 'Buenos Aires', 'Rotterdam', 'Doha',
       'Rio De Janeiro', 'Marseille', 'Acapulco', 'Dubai',
       'Santiago, Chile', 'Indian Wells', 'Miami', 'Estoril', 'Houston',
       'Marrakech', 'Monte Carlo', 'Barcelona', 'Munich', 'Banja Luka',
       'Madrid', 'Rome', 'Geneva', 'Lyon', 'Paris', 's Hertogenbosch',
       'Stuttgart', 'Halle', 'London', 'Mallorca', 'Eastbourne',
       'Wimbledon', 'Gstaad', 'Bastad', 'Newport', 'Atlanta', 'Hamburg',
       'Umag', 'Kitzbuhel', 'Los Cabos, Baja California Sur, Mexico',
       'Washington', 'Toronto', 'Cincinnati', 'Winston-Salem', 'New York',
       'Chengdu', 'Zhuhai', 'Astana', 'Beijing', 'Shanghai', 'Stockholm',
       'Tokyo', 'Antwerp', 'Basel', 'Vienna', 'Metz', 'Sofia', 'Turin',
       'Jeddah'], dtype=object)

Drop duplicate tournament names so you have 1 row for each unique location

In [17]:
atp2023 = atp2023.drop_duplicates(subset=["tourney_id", "tourney_name"]).reset_index(drop=True)
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date
0,2023-9900,Brisbane,Hard,18,A,2023-01-02
1,2023-9900,Sydney,Hard,18,A,2023-01-02
2,2023-9900,Perth,Hard,18,A,2023-01-02
3,2023-2843,Adelaide 1,Hard,32,A,2023-01-02
4,2023-0891,Pune,Hard,32,A,2023-01-02
...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64,M,2023-10-30
65,2023-0341,Metz,Hard,32,A,2023-11-06
66,2023-7434,Sofia,Hard,32,A,2023-11-06
67,2023-0605,Turin,Hard,8,A,2023-11-13


Fix a data error for the level of the Turin event

In [19]:
atp2023.loc[atp2023['tourney_name'] == 'Turin', 'tourney_level'] = 'F'

Make the data in the tourney_level column clearer

In [21]:
atp2023["tourney_level"] = atp2023["tourney_level"].replace({
    "A": "Tour Event",
    "G": "Grand Slam",
    "M": "Masters",
    "F": "Finals"
})

Make a seperate column for the tournament levels expressed as numbers

In [23]:
conditions = [
    atp2023["tourney_level"] == "Grand Slam",
    atp2023["tourney_level"] == "Finals",
    atp2023["tourney_level"] == "Masters",
]
choices = [4, 3, 2]

atp2023["Level Weight"] = np.select(conditions, choices, default=1)

Add a month name and month number column

In [25]:
atp2023["Month Name"] = atp2023["tourney_date"].dt.strftime("%B")
atp2023["Month Number"] = atp2023["tourney_date"].dt.month

Add the word 'draw' into the draw size column and add a draw weight column for sorting

In [27]:
atp2023["draw_size"] = atp2023["draw_size"].astype(str) + " Draw"
conditions = [
    atp2023["draw_size"] == "128 Draw",
    atp2023["draw_size"] == "64 Draw",
    atp2023["draw_size"] == "32 Draw",
    atp2023["draw_size"] == "18 Draw",
]
choices = [5, 4, 3, 2]

atp2023["Draw Weight"] = np.select(conditions, choices, default=1)

Create a dataframe that contains a list of the different locations tournaments are held and the locations for each one.

In [29]:
geolocator = Nominatim(user_agent="ATP_Tour_2023")                    # The file the function is being run on
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)        # Nominatim rate limit

unique_cities = atp2023["tourney_name"]                               # Create a list of all the cities

location_data = {}                                                    # Create dictionary
for city in unique_cities:                                            # Run for loop with geocode function to locate each city 
    location = geocode(city)
    if location:                                                      # If else logic to find latitude and longitude of each city and place in dictionary
        location_data[city] = (location.latitude, location.longitude)
    else:
        print(f"Could not geocode: {city}")                           # Flag any misses
location_data
coord_df = pd.DataFrame.from_dict(                                    # Create df from dictionary populated above 
    location_data, orient="index", columns=["latitude", "longitude"]
)
coord_df.index.name = "tourney_name"                                  # Make the index the tournament name/city and reset the index
coord_df = coord_df.reset_index()
coord_df

,tourney_name,latitude,longitude
0,Brisbane,-27.468962,153.023501
1,Sydney,-33.869844,151.208285
2,Perth,-31.955897,115.860578
3,Adelaide 1,-34.819459,138.504749
4,Pune,18.521374,73.854507
...,...,...,...
63,Vienna,48.208354,16.372504
64,Metz,49.119696,6.176355
65,Sofia,42.697703,23.321736
66,Turin,45.067755,7.682489


Merge this dataframe into the original one

In [31]:
atp2023 = atp2023.merge(coord_df, on="tourney_name", how="left")
atp2023

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,Level Weight,Month Name,Month Number,Draw Weight,latitude,longitude
0,2023-9900,Brisbane,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-33.869844,151.208285
2,2023-9900,Perth,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32 Draw,Tour Event,2023-01-02,1,January,1,3,-34.819459,138.504749
4,2023-0891,Pune,Hard,32 Draw,Tour Event,2023-01-02,1,January,1,3,18.521374,73.854507
...,...,...,...,...,...,...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64 Draw,Masters,2023-10-30,2,October,10,4,48.858890,2.320041
65,2023-0341,Metz,Hard,32 Draw,Tour Event,2023-11-06,1,November,11,3,49.119696,6.176355
66,2023-7434,Sofia,Hard,32 Draw,Tour Event,2023-11-06,1,November,11,3,42.697703,23.321736
67,2023-0605,Turin,Hard,8 Draw,Finals,2023-11-13,3,November,11,1,45.067755,7.682489


Fix up the column names

In [33]:
atp2023 = atp2023.rename(columns={
    "tourney_id": "Tournament ID",
    "tourney_name": "Tournament City",
    "surface": "Surface",
    "draw_size": "Draw Size",
    "tourney_level": "Tournament Level",
    "tourney_date": "Tournament Date",
    "latitude": "Latitude",
    "longitude": "Longitude"
})
atp2023

,Tournament ID,Tournament City,Surface,Draw Size,Tournament Level,Tournament Date,Level Weight,Month Name,Month Number,Draw Weight,Latitude,Longitude
0,2023-9900,Brisbane,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-27.468962,153.023501
1,2023-9900,Sydney,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-33.869844,151.208285
2,2023-9900,Perth,Hard,18 Draw,Tour Event,2023-01-02,1,January,1,2,-31.955897,115.860578
3,2023-2843,Adelaide 1,Hard,32 Draw,Tour Event,2023-01-02,1,January,1,3,-34.819459,138.504749
4,2023-0891,Pune,Hard,32 Draw,Tour Event,2023-01-02,1,January,1,3,18.521374,73.854507
...,...,...,...,...,...,...,...,...,...,...,...,...
64,2023-0352,Paris,Hard,64 Draw,Masters,2023-10-30,2,October,10,4,48.858890,2.320041
65,2023-0341,Metz,Hard,32 Draw,Tour Event,2023-11-06,1,November,11,3,49.119696,6.176355
66,2023-7434,Sofia,Hard,32 Draw,Tour Event,2023-11-06,1,November,11,3,42.697703,23.321736
67,2023-0605,Turin,Hard,8 Draw,Finals,2023-11-13,3,November,11,1,45.067755,7.682489


Export/save as CSV file

In [34]:
atp2023.to_csv("atp_tour_2023.csv", index=False)

In [35]:
atp2023.dtypes

Tournament ID               object
Tournament City             object
Surface                     object
Draw Size                   object
Tournament Level            object
Tournament Date     datetime64[ns]
Level Weight                 int32
Month Name                  object
Month Number                 int32
Draw Weight                  int32
Latitude                   float64
Longitude                  float64
dtype: object